# WhisperX for uploading files

Upload your local files to the Colab Files from the left sidebar.

从左侧将视频音频文件上传到Colab，然后运行即可下载生成好的字幕文件。



In [ ]:
# @title *通用参数/Required settings:**
# Check GPU availability
!nvidia-smi

# @markdown #### **Initial prompt**
# @markdown Prompts can be very helpful for correcting specific words or acronyms that the model often misrecognizes in the audio.
prompt = "This is a technical sharing session about data center networking and network operations, primarily in English with some Chinese. Technical terms include NC (Network Cluster), DCN (Data Center Network), MPLS, MPLS-VPN, BGP, eBGP, iBGP, OSPF, IS-IS, MP-BGP, EVPN, VXLAN, VLAN, VRF, ECMP, LACP, Eth-Trunk, Route Reflector, Route Policy, AS-PATH, Next Hop, Prefix, Core, Aggregation, ToR, Leaf, Spine, DCGW, BMC, BMCAGG, AZ, LLD, MOP, Rollback, Migration, Decommission, Failover, Huawei, Ruijie, CloudEngine, CE6865, NE40E, VRP, eSight, MNet, and CloudMNet."  # @param {type:"string"}

# @markdown #### **Directory Path**
# @markdown where your audio-video files are located from Coloab
directory_path = "/content"  # @param {type:"string"}

# @markdown #### Model
model_size = "large-v3"  # @param ["base", "base.en", "small", "small.en","medium", "medium.en", "large-v1","large-v2","large-v3"]

# @markdown #### Language
language = "auto" # @param ["auto", "en", "zh", "de", "es", "ru", "ko", "fr", "ja", "pt", "tr", "pl", "ca", "nl", "ar", "sv", "it", "id", "hi", "fi", "vi", "he", "uk", "el", "ms", "cs", "ro", "da", "hu", "ta", "no", "th", "ur", "hr", "bg", "lt", "la", "mi", "ml", "cy", "sk", "te", "fa", "lv", "bn", "sr", "az", "sl", "kn", "et", "mk", "br", "eu", "is", "hy", "ne", "mn", "bs", "kk", "sq", "sw", "gl", "mr", "pa", "si", "km", "sn", "yo", "so", "af", "oc", "ka", "be", "tg", "sd", "gu", "am", "yi", "lo", "uz", "fo", "ht", "ps", "tk", "nn", "mt", "sa", "lb", "my", "bo", "tl", "mg", "as", "tt", "haw", "ln", "ha", "ba", "jw", "su"]


# @markdown #### Assign speaker labels
# @markdown Recognize speakers
assign_speaker_lable = False # @param {type:"boolean"}

# @markdown #### Align whisper output
align_whisper_output = True # @param {type:"boolean"}

# Install WhisperX

In [ ]:
# @title Install WhisperX
import subprocess, sys
from importlib.metadata import version, PackageNotFoundError

WHISPERX_VERSION = "3.8.6"

try:
    installed = version("whisperx")
except PackageNotFoundError:
    installed = None

if installed != WHISPERX_VERSION:
    # Use subprocess instead of !pip so Colab does not show the restart-session dialog.
    # Transcription runs as a separate process, so the numpy already loaded in the kernel does not matter.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"whisperx=={WHISPERX_VERSION}"],
        check=True,
    )
print(f"WhisperX {version('whisperx')} ready.")

# Get Subtitle Files using WhisperX

In [ ]:
import os
import subprocess
from google.colab import files, userdata

supported_extensions = ('.mp4', '.wav', '.mp3', '.m4a')
skip_dirs = {'sample_data', '.config'}

def build_command(filename, output_dir):
    cmd = ["whisperx", filename,
           "--model", model_size,
           "--output_dir", output_dir,
           "--print_progress", "True",
           "--verbose", "True"]
    if language != "auto":
        cmd += ["--language", language]
    if prompt:
        cmd += ["--initial_prompt", prompt]
    if align_whisper_output:
        cmd += ["--align_model", "WAV2VEC2_ASR_LARGE_LV60K_960H"]
    if assign_speaker_lable:
        hf_token = userdata.get("HF_TOKEN")
        if not hf_token:
            raise RuntimeError("请在 Colab Secrets 中设置 HF_TOKEN 后再启用说话人识别。")
        cmd += ["--diarize", "--hf_token", hf_token]
    return cmd

def process_file(filename):
    output_dir = os.path.dirname(filename)
    print(f"Transcribing: {filename}")
    subprocess.run(build_command(filename, output_dir), check=True)

    base_filename = os.path.splitext(filename)[0]
    for ext in ('.srt', '.json'):
        path = base_filename + ext
        if os.path.exists(path):
            files.download(path)
        else:
            print(f"Missing output: {path}")

def process():
    for root, dirs, filenames in os.walk(directory_path):
        dirs[:] = [d for d in dirs if d not in skip_dirs]
        for name in filenames:
            if name.lower().endswith(supported_extensions):
                process_file(os.path.join(root, name))

process()